# 🧬 파이썬 객체지향 문법: 상속 (Inheritance)


---

## 🎯 학습 목표

- **코드 중복**이 왜 위험한지 설명하고, 상속으로 해결할 수 있다.
- `class 자식(부모):` 문법으로 클래스를 물려받을 수 있다.
- **오버라이딩(overriding)** 으로 부모 메서드를 덮어쓸 수 있다.
- `super()` 로 부모의 생성자와 메서드를 호출할 수 있다.
- 3차시에서 배운 `_protected` 와 `__private` 가 **상속에서 어떻게 다른지** 직접 확인할 수 있다.
- `ApiClient` 를 부모로 두고 `OpenAIClient`, `GeminiClient` 를 만들 수 있다.



---

## 0. 왜 상속이 필요한가요?

### 😱 문제 상황: 복사-붙여넣기 지옥

AI 서비스를 두 개 연동하려고 클래스를 각각 만들었습니다. 코드를 비교해 보세요.

In [ ]:
# ✍️ 아래 두 클래스를 보고, 중복된 부분이 몇 군데인지 세어 보세요.
class OpenAIClient:
    def __init__(self, api_key):
        self.__api_key = api_key
        self._call_count = 0

    def _mask(self):
        return self.__api_key[:4] + "*" * 6

    def ask(self, prompt):
        self._call_count += 1
        return f"[OpenAI] {prompt}"


class GeminiClient:
    def __init__(self, api_key):
        self.__api_key = api_key
        self._call_count = 0

    def _mask(self):
        return self.__api_key[:4] + "*" * 6

    def ask(self, prompt):
        self._call_count += 1
        return f"[Gemini] {prompt}"


print(OpenAIClient("sk-aaaa1111").ask("안녕"))
print(GeminiClient("gm-bbbb2222").ask("안녕"))

# 📌 중복된 부분은 몇 군데인가요?
# 📌 마스킹 방식(앞 4글자)을 바꿔야 한다면 몇 곳을 고쳐야 하나요?

### 💣 중복이 위험한 이유

- 마스킹 규칙을 바꾸려면 **두 곳(나중엔 열 곳)** 을 모두 고쳐야 합니다.
- 한 곳만 고치고 다른 곳을 잊으면 → **버그가 조용히 숨습니다.**
- 실무 격언: **DRY (Don't Repeat Yourself) — 반복하지 마라.**

### 🧬 비유 ① 가족 유전

- 부모의 혈액형·눈동자 색을 자녀가 물려받습니다. 자녀는 거기에 **자기만의 특징**을 더합니다.
- 부모 = **부모 클래스(상위 클래스, super class)**
- 자녀 = **자식 클래스(하위 클래스, sub class)**

### 🏪 비유 ② 프랜차이즈 본사와 지점

- 본사(부모)가 **레시피와 인테리어 기준**을 정합니다.
- 각 지점(자식)은 그 기준을 그대로 쓰면서, **지점 한정 메뉴**를 추가합니다.
- 본사가 레시피를 바꾸면 **모든 지점에 한 번에 반영**됩니다. ← 이게 상속의 힘!

---

## 1. 기본 상속: 괄호 안에 부모를 적는다

### 📖 문법

```python
class 자식클래스(부모클래스):
    pass
```

- 자식은 부모의 **속성과 메서드를 그대로 물려받습니다.**
- 자식 클래스에 아무것도 안 써도(`pass`) 부모 기능을 다 쓸 수 있습니다.

In [ ]:
# ✍️ 빈칸을 채우세요
class Animal:
    def __init__(self, name):
        self.name = name

    def eat(self):
        print(f"{self.name}: 밥을 먹는다 🍚")

    def sleep(self):
        print(f"{self.name}: 잠을 잔다 😴")


class Dog(___):         # 빈칸: 물려받을 부모 클래스 이름
    ___                 # 빈칸: 내용이 없을 때 쓰는 키워드


d = Dog("초코")
d.eat()
d.sleep()
print(d.name)

### ✅ 체크포인트

- `Dog` 안에는 코드가 한 줄도 없지만 `eat()`, `sleep()`, `name` 을 모두 쓸 수 있습니다.
- 파이썬은 `d.eat()` 를 찾을 때 **① Dog 에서 찾고 → ② 없으면 Animal 에서 찾습니다.**

---

## 2. 자식만의 기능 추가하기

자식 클래스는 부모 기능을 **그대로 쓰면서** 자기만의 메서드를 더할 수 있습니다.

In [ ]:
# ✍️ 직접 작성해 보세요 (위 Animal 클래스를 그대로 사용)
# 1) Dog(Animal) 클래스에 bark() 추가 → "이름: 멍멍! 🐶" 출력
# 2) Cat(Animal) 클래스에 scratch() 추가 → "이름: 긁는다 🐱" 출력
# 3) 각 객체를 만들어 부모 메서드(eat)와 고유 메서드를 모두 호출
# 4) Dog 객체로 scratch() 를 호출하면 어떤 에러가 나는지 try/except 로 확인

---

## 3. 오버라이딩(overriding): 부모 메서드 덮어쓰기

### 📖 개념

- 자식 클래스에서 **부모와 같은 이름의 메서드**를 다시 정의하면, 자식의 것이 **우선 적용**됩니다.
- 이것을 **오버라이딩(재정의)** 이라고 합니다.

### 🍳 비유

- 본사 레시피(부모)가 "라면은 3분"인데, 어떤 지점(자식)은 "우리는 4분 끓인다"로 **덮어씁니다.**
- 그 지점에서는 **지점 레시피**가 적용됩니다.

In [ ]:
# ✍️ 빈칸을 채우세요
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        print(f"{self.name}: ...")


class Dog(Animal):
    def ___(self):                        # 빈칸: 부모와 "같은 이름"의 메서드
        print(f"{self.name}: 멍멍!")


class Cat(Animal):
    def ___(self):                        # 빈칸
        print(f"{self.name}: 야옹~")


Animal("동물").speak()
Dog("초코").speak()
Cat("나비").speak()

### 💡 이 코드가  **다형성**의 기본 입니다

같은 `speak()` 라는 **하나의 명령**으로 객체마다 **다른 결과**가 나왔죠? 이걸 본격적으로 활용합니다.

---

## 4. `super()`: 부모의 기능을 불러와 재사용하기

### 😵 문제: 오버라이딩하면 부모 코드가 사라진다

자식이 `__init__` 을 새로 쓰면 **부모의 `__init__` 은 실행되지 않습니다.**

In [ ]:
# ✍️ 아래 코드를 실행하고, 왜 에러가 나는지 적어 보세요.
class Animal:
    def __init__(self, name):
        self.name = name
        print("Animal 생성자 실행")


class Dog(Animal):
    def __init__(self, name, breed):
        self.breed = breed


d = Dog("초코", "푸들")
print("품종:", d.breed)

try:
    print("이름:", d.name)
except AttributeError as e:
    print("❌ 에러:", e)

# 📌 왜 name 이 없을까요?

### 🔑 해결책: `super().__init__(...)`

- `super()` = **"내 부모"** 를 가리키는 말
- `super().__init__(name)` → **부모의 생성자를 먼저 실행**해서 기본 세팅을 끝냅니다.
- 그 뒤에 자식만의 속성을 추가합니다.

### 🏗️ 비유

- 집을 지을 때 **기초 공사(부모 생성자)** 를 먼저 하고, 그 위에 **내 방 인테리어(자식 속성)** 를 합니다.
- 기초 공사를 건너뛰면 집이 무너집니다.

### ⚠️ 순서 규칙

`super().__init__()` 는 **자식 `__init__` 의 맨 윗줄**에 쓰는 것을 권장합니다.

In [ ]:
# ✍️ 빈칸을 채우세요
class Animal:
    def __init__(self, name):
        self.name = name

    def info(self):
        print(f"이름: {self.name}")


class Dog(Animal):
    def __init__(self, name, breed):
        ___().__init__(___)              # 빈칸 2개: 부모 생성자 호출 + 넘길 값
        self.breed = breed

    def info(self):
        ___().info()                      # 빈칸: 부모의 info 재사용
        print(f"품종: {self.breed}")


d = Dog("초코", "푸들")
d.info()
# 기대 출력: 이름: 초코 / 품종: 푸들

---

## 5. 🔗 복습 연결: 상속에서 `_` 와 `__` 는 다르게 동작합니다

 배운 내용을 기억하세요.

| 표기 | 자식 클래스에서 사용 |
|---|---|
| `self._prot` (protected) | ✅ **쓸 수 있다** |
| `self.__priv` (private) | ❌ **쓸 수 없다** (이름 맹글링 때문) |

**이것이 바로 protected 가 존재하는 이유입니다.**

In [ ]:
# ✍️ 직접 작성해 보세요
# 1) Parent 클래스: __init__ 에서 pub(public), _prot(protected), __priv(private) 저장
# 2) Child(Parent) 클래스의 test() 에서 세 개를 모두 출력 시도
#    (private 은 try / except AttributeError 로 감싸기)
# 3) c.__dict__.keys() 를 출력해 실제 저장된 이름 확인
# 📌 private 접근이 실패한 이유는?

> 💡 **정리**: 자식에게 물려줄 내부 데이터는 **`_` (protected)**, 아무에게도 안 보여줄 것은 **`__` (private)** 입니다.
> 그래서 실무 클래스에서는 `_call_count` 처럼 밑줄 1개를 자주 봅니다.

---

## 6. 관계 확인하기: `isinstance()` 와 `issubclass()`

| 함수 | 질문 | 예시 |
|---|---|---|
| `isinstance(객체, 클래스)` | "이 **객체**가 그 클래스에서 나왔나요?" | `isinstance(d, Animal)` |
| `issubclass(자식, 부모)` | "이 **클래스**가 그 클래스를 물려받았나요?" | `issubclass(Dog, Animal)` |

### 💡 핵심: 자식 객체는 부모 타입으로도 인정됩니다

"초코는 개다"도 맞고 "초코는 동물이다"도 맞죠. 이것이 다형성의 기반입니다.

In [ ]:
# ✍️ 빈칸을 채우세요
class Animal: pass
class Dog(Animal): pass
class Cat(Animal): pass

d = Dog()

print("Dog 객체인가? ", ___(d, Dog))          # 빈칸: 객체-클래스 관계 확인 함수
print("Animal 객체인가?", ___(d, Animal))      # 결과를 예측해 보세요!
print("Cat 객체인가?  ", ___(d, Cat))
print()
print("Dog 는 Animal 의 자식?", ___(Dog, Animal))   # 빈칸: 클래스-클래스 관계 확인 함수
print("Animal 은 Dog 의 자식?", ___(Animal, Dog))

### 🔍 한 걸음 더: 파이썬은 메서드를 어떤 순서로 찾나요? (MRO)

`__mro__` (Method Resolution Order) 는 파이썬이 메서드를 찾아가는 **순서 목록**입니다.

In [ ]:
# ✍️ 직접 확인해 보세요
# 1) A(hello 메서드 있음) → B(A) → C(B) 3단계 상속 만들기
# 2) C().hello() 를 호출 → 어디의 hello 가 실행되나요?
# 3) C.__mro__ 를 for 문으로 돌며 cls.__name__ 출력
# 📌 탐색 순서:

---

## 7. 🛠️ 단계별 실습 과제

### 실습 1 교통수단 클래스

**요구사항**

- 부모 `Vehicle`: `__init__(self, name, wheels)`, `info()` → `"이름: 바퀴 N개"` 출력
- 자식 `Car(Vehicle)`: 생성 시 바퀴는 **4개 고정**, `honk()` → `"빵빵!"` 출력
- 자식 `Bicycle(Vehicle)`: 생성 시 바퀴는 **2개 고정**, `ring()` → `"따르릉!"` 출력
- `super()` 를 사용해 부모 생성자를 호출할 것

In [ ]:
# ✍️ 실습 1: 빈칸을 채워 완성하세요
class Vehicle:
    def __init__(self, name, wheels):
        self.name = name
        self.wheels = wheels

    def info(self):
        print(f"{self.name}: 바퀴 {self.wheels}개")


class Car(___):                          # 빈칸: 부모 클래스
    def __init__(self, name):
        super().__init__(name, ___)      # 빈칸: 자동차 바퀴 개수
    def honk(self):
        print("빵빵!")


class Bicycle(Vehicle):
    def __init__(self, name):
        ___                              # 빈칸: super() 로 부모 생성자 호출 (바퀴 2개)
    def ring(self):
        print("따르릉!")


car = Car("소나타")
bike = Bicycle("자전거")

car.info();  car.honk()      # 기대: 소나타: 바퀴 4개 / 빵빵!
bike.info(); bike.ring()     # 기대: 자전거: 바퀴 2개 / 따르릉!

### 실습 2 중복 제거! `ApiClient` 상속 구조 🔑

> **중복 코드를 상속으로 리팩터링합니다.**  `ApiClient` 가 부모가 됩니다.

**요구사항**

- 부모 `ApiClient`
  - `__init__(self, service_name, api_key)`: `service_name`(public), `__api_key`(private), `_call_count = 0`(protected)
  - `get_masked_key()`: 앞 4글자 + 나머지는 `*`
  - `ask(self, prompt)`: `_call_count` 1 증가 후 `"[서비스명] 기본 응답: 프롬프트"` 리턴
  - `get_call_count()`: 호출 횟수 리턴
- 자식 `OpenAIClient(ApiClient)`
  - 생성 시 서비스명은 **"OpenAI" 고정**, `model` 속성 추가 (기본값 `"gpt-4"`)
  - `ask()` 를 오버라이딩 → `"[OpenAI/모델명] 프롬프트"` 리턴 (호출 횟수 증가는 유지!)
- 자식 `GeminiClient(ApiClient)`
  - 생성 시 서비스명은 **"Gemini" 고정**
  - `ask()` 오버라이딩 → `"[Gemini] 프롬프트 (안전필터 통과)"` 리턴

**힌트**: 자식 `ask()` 에서 호출 횟수를 세려면 `self._call_count += 1` 을 쓰거나 `super().ask(prompt)` 를 활용하세요.

In [ ]:
# ✍️ 실습 2: 요구사항을 보고 상속 구조를 완성하세요
class ApiClient:
    def __init__(self, service_name, api_key):
        pass   # TODO 1: service_name(public), __api_key(private), _call_count=0(protected)

    def get_masked_key(self):
        pass   # TODO 2: 앞 4글자 + 나머지 '*'

    def ask(self, prompt):
        pass   # TODO 3: _call_count 1 증가 후 "[서비스명] 기본 응답: 프롬프트" return

    def get_call_count(self):
        pass   # TODO 4: 호출 횟수 return


class OpenAIClient(ApiClient):
    def __init__(self, api_key, model="gpt-4"):
        pass   # TODO 5: super() 로 서비스명 "OpenAI" 고정 + self.model 저장

    def ask(self, prompt):
        pass   # TODO 6: 호출 횟수 증가 + "[OpenAI/모델명] 프롬프트" return


class GeminiClient(ApiClient):
    def __init__(self, api_key):
        pass   # TODO 7: super() 로 서비스명 "Gemini" 고정

    def ask(self, prompt):
        pass   # TODO 8: 호출 횟수 증가 + "[Gemini] 프롬프트 (안전필터 통과)" return


# 테스트 코드 (수정 금지)
gpt = OpenAIClient("sk-aaaa1111", model="gpt-4o")
gem = GeminiClient("gm-bbbb2222")

print(gpt.ask("파이썬이 뭐야?"))
print(gpt.ask("한 줄 요약해줘"))
print(gem.ask("파이썬이 뭐야?"))

print("키 마스킹:", gpt.get_masked_key())       # 기대: sk-a 뒤가 모두 * 로 가려진 형태
print("GPT 호출수:", gpt.get_call_count())      # 기대: 2
print("Gemini 호출수:", gem.get_call_count())   # 기대: 1
print("둘 다 ApiClient 인가?", isinstance(gpt, ApiClient), isinstance(gem, ApiClient))

### 실습 3 [중급 · 포트폴리오용 · 25분] 특화 챗봇 만들기 🤖

> 3차시 `ChatBot` 을 부모로 두고 **번역봇**과 **요약봇**을 만듭니다.

**요구사항**

- 부모 `ChatBot`
  - `__init__(self, name)`: `name`(public), `__history = []`(private)
  - `say(self, message)`: 기록에 저장하고 `_reply(message)` 결과를 출력
  - `_reply(self, message)`: **protected 메서드**, 기본 응답 `"'메시지' 라고 하셨네요"` 리턴
  - `count()`: 기록 개수 리턴
- 자식 `TranslateBot(ChatBot)`: `_reply()` 오버라이딩 → `"[번역] 메시지 → (영어 번역 결과)"`
- 자식 `SummaryBot(ChatBot)`: `_reply()` 오버라이딩 → `"[요약] 메시지 (앞 5글자만)..."`
- 세 봇을 모두 만들어 같은 문장을 `say()` 해 보고, 기록이 각각 쌓이는지 확인

**포인트**: 부모의 `say()` 는 **한 번만** 작성했는데, 자식마다 다른 응답이 나옵니다. 이게 상속의 가치입니다.

In [ ]:
# ✍️ 실습 3: ChatBot 상속 구조를 직접 작성하세요 (골격만 제공)
class ChatBot:
    def __init__(self, name):
        pass   # TODO: name(public), __history=[](private)

    def _reply(self, message):
        pass   # TODO: 기본 응답 "'메시지' 라고 하셨네요" return

    def say(self, message):
        pass   # TODO: 기록에 저장 → "[이름] {self._reply(message)}" 출력

    def count(self):
        pass   # TODO: 기록 개수 return


class TranslateBot(ChatBot):
    pass   # TODO: _reply() 오버라이딩 → "[번역] 메시지 → (영어 번역 결과)"


class SummaryBot(ChatBot):
    pass   # TODO: _reply() 오버라이딩 → "[요약] 앞5글자..."  (힌트: message[:5])


# 테스트 코드 (수정 금지)
basic = ChatBot("기본봇")
trans = TranslateBot("번역봇")
summ  = SummaryBot("요약봇")

sentence = "파이썬 객체지향은 재미있습니다"
basic.say(sentence)
trans.say(sentence)
summ.say(sentence)

trans.say("한 번 더 번역해줘")
print("번역봇 기록:", trans.count(), "개")   # 기대: 2
print("요약봇 기록:", summ.count(), "개")    # 기대: 1

---

## 8. 📝 이해도 확인 퀴즈

**Q1.** `class Dog(Animal):` 에서 부모와 자식은 각각 무엇인가요?
- 📌 내 답:

**Q2.** 자식이 부모와 같은 이름의 메서드를 다시 정의하는 것을 무엇이라고 하나요?
- 📌 내 답:

**Q3.** `super().__init__(name)` 을 쓰지 않으면 어떤 문제가 생기나요?
- 📌 내 답:

**Q4.** 부모의 `_prot` 와 `__priv` 중 자식이 쓸 수 있는 것은 무엇이며, 그 이유는?
- 📌 내 답:

**Q5.** 아래 코드의 출력을 **먼저 예측**한 뒤 다음 셀에서 확인하세요.
- 📌 내 예측:

In [ ]:
class A:
    def __init__(self):
        print("A 생성")
    def hi(self):
        print("A 의 hi")

class B(A):
    def __init__(self):
        super().__init__()
        print("B 생성")

class C(B):
    def hi(self):
        print("C 의 hi")
        super().hi()

c = C()
c.hi()
print(isinstance(c, A))

---

## 9. 📊 상속  정리

| 개념 | 문법 | 비유 |
|---|---|---|
| 상속 | `class 자식(부모):` | 부모의 유전자를 물려받기 |
| 기능 추가 | 자식에 새 메서드 정의 | 지점 한정 메뉴 |
| 오버라이딩 | 부모와 **같은 이름**의 메서드 | 지점만의 레시피로 덮어쓰기 |
| `super()` | `super().__init__(...)` | 기초 공사 먼저 하고 내 방 꾸미기 |
| `isinstance` | `isinstance(d, Animal)` | "초코는 동물이기도 하다" |

### ⚠️ 상속을 남용하지 마세요

- 상속은 **"~는 ~이다 (is-a)"** 관계일 때만 씁니다. (개는 동물이다 ✅)
- **"~는 ~를 가지고 있다 (has-a)"** 관계는 상속이 아니라 **포함(composition)** 을 씁니다. (자동차는 엔진을 가지고 있다 → 상속 ❌)
- 상속 단계가 4~5단으로 깊어지면 **추적이 어려워집니다.** 2~3단을 권장합니다.

---



**오늘 배운 것**

- 상속은 **중복 제거**와 **일관성 유지**를 위한 도구다.
- 자식은 부모를 물려받고, **오버라이딩**으로 바꿔 쓸 수 있다.
- `super()` 로 부모의 기능을 **재사용**한다.
- `_protected` 는 자식이 쓸 수 있고 `__private` 는 쓸 수 없다.


> 🙌 상속은 "코드를 줄이는 기술"이 아니라 **"변경에 강한 구조를 만드는 기술"** 입니다. 지금은 코드가 줄어드는 재미만 느껴도 충분합니다!